# PBAIH12 作業｜這個高分的命中可信嗎？
**智慧醫療大數據實作與分析**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/PBAIH/blob/main/homework/PBAIH12_HW.ipynb)

**個人作業**，兩週內完成（約 1 小時的工作量，**比 PBAIH11 輕**）。**期限：發布日起兩週**（確切日期與時間見 Im@TMU 公告）。

**開始前**：先「檔案 → 在雲端硬碟中儲存複本」，在自己的複本上作答。
**繳交（兩件事都要做）**：
1. 完成後「檔案 → 下載 → 下載 .ipynb」，上傳到 Im@TMU；檔名 `PBAIH12_HW_學號_姓名.ipynb`。
2. 在 Colab 右上角「共用」，把 `syw@tmu.edu.tw` 加為「**檢視者**」，再把這份複本的連結貼到下面「學生資訊」格的 `COLAB_LINK`。

**繳交前**：「執行階段 → 重新啟動並全部執行」，確認從頭到尾沒有錯誤。（第 3 部分的答案要先寫好，才執行「揭曉」那一格。）

### 情境
你是代謝體核心實驗室的分析助理。研究者從一批樣品挑出 3 張**未知的 MS/MS 譜**，請你去公開譜庫（MassBank）比對，並**判斷每一個命中能不能報告**。譜庫裡的譜都來自**別的實驗室**；這 3 張未知譜是同一個實驗室測的，真正的化合物只有最後的「揭曉」那一格才會告訴你。

### 每個人的未知譜都不一樣
程式會用你的**學號**抽出你的 3 張未知譜，所以每個人拿到的不同，但**判斷的邏輯是一樣的**。請在第一格填入自己的學號，不要用別人的學號。

### 評分標準（100 分）
| 項目 | 配分 | 評分重點 |
|---|---|---|
| 2. 比對與門檻（程式） | 20 | 補完 3 個地方：對每個未知找候選、設定你的分數門檻和匹配峰數門檻；結果能跑出來 |
| 3. 逐一判讀 | 45 | 每個未知 15 分：報告或不報告、信心等級、用**數字**（分數、匹配峰數、和其他化合物的差距）說明理由 |
| 4. 對照答案後的反思 | 35 | 說明哪個未知「只看分數」會判錯；寫出至少兩項你會再做的確認 |
| GenAI 使用說明 | 必填（不計分） | 如實揭露不扣分；使用了卻未揭露，作業成績上限 60 分 |

> 💡 程式寫法可以參考課堂的實作二 `PBAIH12B`（Part 2 的 `search_one`）：標示「**提供，不需修改**」的程式格都是從那裡整理來的，你只要補完 `# TODO` 那幾行。
> 💡 「信心等級」請對照投影片〈信心等級〉那一頁，選你認為合適的一級。
> 💡 卡住的順序：① 錯誤訊息的最後一行 → ② Colab 的 Gemini（按「解釋錯誤」，請它用中文解釋）→ ③ 實作二的對應段落 → ④ 上課時舉手問。
> 💡 圖表的文字用英文，因為 Colab 預設沒有中文字型。

## 學生資訊（必填）
填好三個欄位後執行這一格。`COLAB_LINK` 是你這份複本的分享連結（先按右上角「共用」把 `syw@tmu.edu.tw` 加為檢視者，再按「複製連結」）。

In [ ]:
STUDENT_ID = ""     # 你的學號，例如 "B812113001"（要加引號）
NAME = ""           # 你的姓名
COLAB_LINK = ""     # 這份 Colab 複本的分享連結（https://colab.research.google.com/...）

import hashlib
if not str(STUDENT_ID).strip():
    raise ValueError("請先在上面填入你的學號 STUDENT_ID（例如 \"B812113001\"），再執行這一格。")
STUDENT_ID = str(STUDENT_ID).strip()
SEED = int(hashlib.sha256(STUDENT_ID.encode("utf-8")).hexdigest()[:8], 16)   # 學號 → 固定的隨機種子
print("學號：", STUDENT_ID, "｜姓名：", NAME or "（尚未填寫）", "｜你的 seed：", SEED)
if not str(COLAB_LINK).startswith("https://colab.research.google.com/"):
    print("⚠️ 提醒：COLAB_LINK 還不是 Colab 連結；繳交前記得貼上，並把 syw@tmu.edu.tw 加為檢視者。")

## 0. 環境與資料（提供，不需修改）
和實作二的 Part 0 一樣：依序執行 0-a（安裝）和 0-b（載入套件與資料）。若 0-a 要你重新啟動工作階段，照做後**直接執行 0-b**。安裝或下載失敗時，照實作二 Part 0.1 的表處理。

In [ ]:
# 0-a 安裝 matchms（只在 Colab 上需要；上課一開始先執行這一格）
import sys
import importlib.metadata as md

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q matchms==1.0.0")

# 安裝可能會升級 numpy／pandas／matplotlib；如果這些套件在安裝前已經被載入，記憶體裡還是舊版，要重新啟動才會換新
stale = [p for p in ("numpy", "pandas", "matplotlib") if p in sys.modules and md.version(p) != sys.modules[p].__version__]
if stale:
    print("⚠️ 這些套件已升級，但記憶體裡還是舊版：", stale)
    print("請按上方選單「執行階段 → 重新啟動工作階段」，然後【直接執行 0-b】（不用再執行 0-a）。")
else:
    print("安裝完成，請執行 0-b。")

In [ ]:
# 0-b 載入套件、取得資料、讀入譜庫子集
import os, sys, glob, zipfile, logging
import importlib.metadata as md

IN_COLAB = "google.colab" in sys.modules
stale = [p for p in ("numpy", "pandas", "matplotlib") if p in sys.modules and md.version(p) != sys.modules[p].__version__]
if stale:
    raise RuntimeError(f"{stale} 的版本在安裝後改變了。請按「執行階段 → 重新啟動工作階段」，再重新執行這一格。")

# 取得資料：優先順序 = 環境變數 PBAIH12_DATA_DIR → 已下載（或手動放好）的資料夾 → 從課程 GitHub 下載
DATA_URL = "https://github.com/wang3yuan/PBAIH/releases/download/data-v2/PBAIH12_data.zip"
DATA_ROOT = "pbaih12_data"

def find_data_dir(*roots):
    """在這些資料夾底下找含有 massbank_pos_ms2.parquet 的資料夾。"""
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, "**", "massbank_pos_ms2.parquet"), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    return None

DATA_DIR = os.environ.get("PBAIH12_DATA_DIR") or find_data_dir(DATA_ROOT, ".")
if DATA_DIR is None:
    import urllib.request
    os.makedirs(DATA_ROOT, exist_ok=True)
    zip_path = os.path.join(DATA_ROOT, "PBAIH12_data.zip")
    print("下載中（約 10 MB）...")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_ROOT)
    DATA_DIR = find_data_dir(DATA_ROOT)
assert DATA_DIR is not None, "找不到資料，請看 Part 0.1"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message=".*hdf5plugin.*")   # matchms 內部套件的無害提示
import matchms
from matchms import Spectrum
from matchms.similarity import CosineGreedy
logging.getLogger("matchms").setLevel(logging.ERROR)   # 關掉 matchms 的警告訊息

for name, mod in [("matchms", matchms), ("numpy", np), ("pandas", pd), ("matplotlib", matplotlib)]:
    print(f"{name:11s} {mod.__version__}")
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| 本機")

lib = pd.read_parquet(os.path.join(DATA_DIR, "massbank_pos_ms2.parquet"))
meta_ms2 = pd.read_parquet(os.path.join(DATA_DIR, "massbank_ms2_meta.parquet"))
print(f"譜庫子集：{len(lib):,} 筆 MS/MS、{lib.ik14.nunique():,} 個化合物、{lib.contributor.nunique()} 個貢獻者")
print("授權：", lib.license.value_counts().to_dict())


# 前處理：把一筆紀錄變成 matchms 的 Spectrum。兩個步驟都寫在這裡，不交給套件預設去做：
#   (1) 先移除前驅物附近的峰（m/z ≥ 前驅物 m/z − 1.6）：沒有碎裂的前驅物離子沒有結構資訊，而且常常是最大的峰
#   (2) 再去掉低於（剩下的）基峰 1% 的峰，當作雜訊
PREC_OFFSET = -1.6
def get_spec(r, min_rel=0.01):
    mz = np.asarray(r.mz, dtype=float); it = np.asarray(r.intensity, dtype=float)
    keep = mz < r.precursor_mz + PREC_OFFSET
    mz, it = mz[keep], it[keep]
    if len(mz):
        keep = it >= min_rel * it.max()
        mz, it = mz[keep], it[keep]
    return Spectrum(mz=mz, intensities=it, metadata={"precursor_mz": float(r.precursor_mz)})

## 1. 你的三張未知譜（提供，不需修改）
這一格依你的學號抽出 3 張未知譜，並建好「其他實驗室」的譜庫（和實作二 Part 1 的做法相同：查詢的實驗室不放進譜庫）。處理方式也和實作二相同：前驅物附近的峰先移除，再去掉低於基峰 1% 的峰。

下面只顯示每張未知譜的**測量條件**，不會顯示它是誰。

In [ ]:
# 1 抽出你的三張未知譜（提供，不需修改）
QUERY_LAB = "BAFG"            # 這 3 張未知譜都來自這個實驗室；譜庫放其他實驗室的譜
MS1_TOL, MS2_TOL = 0.01, 0.025
POOLS = {"A": ["MSBNK-BAFG-CSL2311092996", "MSBNK-BAFG-CSL2311094998", "MSBNK-BAFG-CSL2311095999", "MSBNK-BAFG-CSL23111015295", "MSBNK-BAFG-CSL23111016501", "MSBNK-BAFG-CSL23111017391", "MSBNK-BAFG-CSL23111017408", "MSBNK-BAFG-CSL23111017441", "MSBNK-BAFG-CSL23111017568", "MSBNK-BAFG-CSL23111017756", "MSBNK-BAFG-CSL23111027399", "MSBNK-BAFG-CSL23111027469", "MSBNK-BAFG-CSL2311108063", "MSBNK-BAFG-CSL2311108844", "MSBNK-BAFG-CSL26052937563"], "B": ["MSBNK-BAFG-CSL2311094002", "MSBNK-BAFG-CSL2311094672", "MSBNK-BAFG-CSL2311094868", "MSBNK-BAFG-CSL2311095612", "MSBNK-BAFG-CSL23111011714", "MSBNK-BAFG-CSL23111013124", "MSBNK-BAFG-CSL23111014100", "MSBNK-BAFG-CSL23111016330", "MSBNK-BAFG-CSL23111016897", "MSBNK-BAFG-CSL2311106675", "MSBNK-BAFG-CSL2311107351", "MSBNK-BAFG-CSL2311107901", "MSBNK-BAFG-CSL2311108410", "MSBNK-BAFG-CSL2311108573", "MSBNK-BAFG-CSL2311109961"], "C": ["MSBNK-BAFG-CSL2311093169", "MSBNK-BAFG-CSL2311094489", "MSBNK-BAFG-CSL2311096327", "MSBNK-BAFG-CSL23111011435", "MSBNK-BAFG-CSL23111011636", "MSBNK-BAFG-CSL23111012139", "MSBNK-BAFG-CSL23111015644", "MSBNK-BAFG-CSL23111016739", "MSBNK-BAFG-CSL23111027381", "MSBNK-BAFG-CSL2311106561", "MSBNK-BAFG-CSL2311107551", "MSBNK-BAFG-CSL2311107663", "MSBNK-BAFG-CSL2311107782", "MSBNK-BAFG-CSL2311108175", "MSBNK-BAFG-CSL2311109347"]}
rng = np.random.default_rng(SEED)
picks = [(kind, str(rng.choice(POOLS[kind]))) for kind in "ABC"]
picks = [picks[i] for i in rng.permutation(3)]          # 順序打亂：你不能從「第幾個」猜出是哪一種情況
kinds = [k for k, _ in picks]                            # 揭曉用，現在先不要看

query = pd.concat([lib[lib.accession == acc] for _, acc in picks]).reset_index(drop=True)
ref = lib[lib.contributor != QUERY_LAB].sort_values("precursor_mz").reset_index(drop=True)
ref_specs = [get_spec(r) for _, r in ref.iterrows()]
ref_mz = ref.precursor_mz.values
q_specs = [get_spec(r) for _, r in query.iterrows()]

info = pd.DataFrame({"未知": [1, 2, 3], "前驅物 m/z": query.precursor_mz.round(4),
                     "碰撞能量": query.collision_energy, "儀器": query.instrument_type,
                     "前處理後的峰數": [len(s.peaks.mz) for s in q_specs]})
display(info.set_index("未知"))
print(f"譜庫：其他 {ref.contributor.nunique()} 個實驗室的 {len(ref):,} 張譜")

## 2. 比對與門檻（程式，20 分）
**2-1　找候選（提供，不需修改）**：`search_one` 和實作二 Part 2 相同：用前驅物 m/z 篩出候選，算每個候選的分數和**對上的峰數**，由高到低排序；至少對上 1 個碎片峰才算候選。

In [ ]:
# 2-1 找候選的函式（提供，不需修改；同實作二 Part 2）
def search_one(qi, sim, topn=5):
    q = q_specs[qi]
    pmz = query.precursor_mz[qi]
    lo, hi = np.searchsorted(ref_mz, [pmz - MS1_TOL, pmz + MS1_TOL])
    hits = []
    for j in range(lo, hi):
        res = sim.pair(ref_specs[j], q)
        if int(res["matches"]) < 1:
            continue
        hits.append((float(res["score"]), int(res["matches"]), ref.ik14[j], j))
    hits.sort(reverse=True)
    return hits[:topn]

sim = CosineGreedy(tolerance=MS2_TOL, mz_power=1, intensity_power=0.5, remove_precursor=False)

def plot_mirror(q, l, tol=0.025, title=""):
    fig, ax = plt.subplots(figsize=(7, 3.4))
    lm, li, qm = l.peaks.mz, l.peaks.intensities, q.peaks.mz
    for m, i in zip(qm, q.peaks.intensities):
        ax.vlines(m, 0, i, color="#EE6C4D" if np.any(np.abs(lm - m) <= tol) else "#8D99AE", lw=1.5)
    for m, i in zip(lm, li):
        ax.vlines(m, 0, -i, color="#EE6C4D" if np.any(np.abs(qm - m) <= tol) else "#8D99AE", lw=1.5)
    ax.axhline(0, color="k", lw=0.6); ax.set_ylim(-1.1, 1.1); ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity")
    ax.plot([], [], color="#EE6C4D", lw=2, label="matched"); ax.plot([], [], color="#8D99AE", lw=2, label="not matched")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.set_title(title, fontsize=9); plt.tight_layout(); plt.show()
print("函式已定義")

**2-2　設定你的門檻（補完兩行）**：實作二 Part 4 看過，只用「分數」會有假命中，還要加「匹配峰數」。請設定**你自己**的兩個門檻：第 1 名的分數至少多少、對上的峰至少幾個，你才願意「報告」這個命中。（沒有標準答案，但要能在第 3 部分說明理由。）

In [ ]:
MIN_SCORE = ____      # TODO：分數門檻（0 到 1 之間的數字）
MIN_MATCHES = ____      # TODO：匹配峰數門檻（整數）

<details><summary>💡 練習 2 提示（先自己試，卡住再打開）</summary>

沒有唯一答案。實作二 Part 4 用過「分數 ≥ 0.7 且匹配峰 ≥ 3」；你可以照用，也可以自己調嚴或調鬆，但要能說明為什麼。

<details><summary>參考答案</summary>

```python
MIN_SCORE = 0.7
MIN_MATCHES = 3
```
</details>
</details>

**2-3　對三張未知譜找候選（補完一行）**：用 `search_one` 找第 `i` 張未知譜的候選（前 5 名）。其餘的表格、判斷和鏡像圖已經寫好。

In [ ]:
for i in range(3):
    top = ____        # TODO：用 search_one 找第 i 張未知譜的候選
    print(f"════ 未知 {i + 1}：前驅物 m/z {query.precursor_mz[i]:.4f}，CE {query.collision_energy[i]}，前處理後 {len(q_specs[i].peaks.mz)} 個峰 ════")
    if not top:
        print("沒有任何候選（前驅物範圍內沒有譜，或沒有碎片對上）\n")
        continue
    show = pd.DataFrame([{"rank": k + 1, "name": ref["name"][j], "formula": ref.formula[j], "score": s, "matches": m}
                         for k, (s, m, ik, j) in enumerate(top)])
    display(show.round(3).set_index("rank"))
    s1, m1, ik1, j1 = top[0]
    other = [t[0] for t in top if t[2] != ik1]           # 第 1 名以外的「不同化合物」候選的分數
    print(f"第 1 名：分數 {s1:.3f}，對上 {m1} 個峰；第一個不同化合物的候選分數："
          + (f"{other[0]:.3f}" if other else "（沒有，前 5 名都是同一個化合物）"))
    print("依你的門檻：", "✅ 通過" if (s1 >= MIN_SCORE and m1 >= MIN_MATCHES) else "❌ 沒通過")
    plot_mirror(q_specs[i], ref_specs[j1], title=f"unknown {i + 1} (top) vs rank 1: {ref['name'][j1]} (bottom)   score {s1:.3f}, {m1} matched peaks")

<details><summary>💡 練習 3 提示（先自己試，卡住再打開）</summary>

`search_one(qi, sim)` 的 `qi` 是第幾張未知譜（0、1、2），這裡迴圈變數就是 `i`。

<details><summary>參考答案</summary>

```python
top = search_one(i, sim)
```
</details>
</details>

## 3. 逐一判讀（45 分，每個未知 15 分）
根據上面的表格和鏡像圖，對**每一張**未知譜回答：
1. **報告或不報告**第 1 名這個命中？
2. **信心等級**：對照投影片〈信心等級〉，選你認為合適的一級。
3. **理由**：用**數字**說明（分數、對上幾個峰、第 1 名和「其他不同化合物」的分數差多少、鏡像圖上對上的是哪些峰）。2–3 句即可。

> 注意：你的「報告／不報告」不一定要和第 2 部分的門檻判斷一樣；如果不一樣，請說明為什麼。

> **未知 1**
> - 報告／不報告：
> - 信心等級：
> - 理由（用數字）：

> **未知 2**
> - 報告／不報告：
> - 信心等級：
> - 理由（用數字）：

> **未知 3**
> - 報告／不報告：
> - 信心等級：
> - 理由（用數字）：

## 4. 揭曉與反思（35 分）
**先把第 3 部分三個未知都寫好，再執行下面的「揭曉」。** 揭曉會顯示每張未知譜的真正化合物（以 InChIKey 前 14 碼判斷是不是同一個化合物），以及你的第 1 名是對是錯。

In [ ]:
# 揭曉（提供，不需修改）：寫完第 3 部分再執行
LABEL = {"A": "譜庫裡有答案，第 1 名正確",
         "B": "譜庫裡有答案，但第 1 名是別的化合物（高分的假命中）",
         "C": "譜庫裡沒有答案，第 1 名只是長得像（庫外假命中）"}
for i in range(3):
    top = search_one(i, sim)
    s1, m1, ik1, j1 = top[0]
    print(f"未知 {i + 1}：真正的化合物是 {query['name'][i]}（{query.formula[i]}）")
    print(f"        你的第 1 名：{ref['name'][j1]}（分數 {s1:.3f}，對上 {m1} 個峰）→ {LABEL[kinds[i]]}\n")

**(a)**（20 分）對照答案：哪一個（或哪幾個）未知，「只看分數」會讓你判斷錯？用一句話說明：它的分數是多少、為什麼分數沒有發現問題（可以想：同分異構物、匹配峰數、庫裡根本沒有答案）。

> 你的回答：

**(b)**（15 分）如果你要把其中一個命中寫進報告，除了譜庫比對的分數，你**還會再做哪些確認**？至少寫兩項，每項說明「做什麼、看什麼結果、怎麼判斷」（例如：用標準品在同一個儀器上測、比對 RT、檢查 MS1 的同位素分布、換不同碰撞能量再測……）。

> 你的回答：

---
## GenAI 使用說明（必填，不計分）
依課程〈生成式 AI 使用指引〉（與「智慧醫療與機器學習基礎」相同）：
- 可以使用 GenAI（例如 Colab 的 Gemini、ChatGPT、Claude），但**要揭露**。**如實揭露不會扣分**；使用了卻未揭露，作業成績上限 60 分。
- 本作業用的是**公開資料**（MassBank），可以給 AI 看；但**不可上傳任何個人資料**。
- **不可透過網路使用中國營運的 AI 服務**（例如 DeepSeek、通義千問（Qwen）、Kimi、豆包、文心一言），不論網頁、App 或 API。
- AI 給的程式要能自己解釋每一行；AI 提供的文獻、數字要自己找到原始出處查證。

沒有使用也請寫「本次作業未使用 GenAI」。

> 本次作業是否使用 GenAI：
>
> 用了哪些工具（例如 Colab 的 Gemini、ChatGPT）：
>
> 用在哪一題、怎麼問（例如：第 3 部分請 Gemini 解釋什麼是同分異構物）：
>
> 怎麼查證、有沒有修改它給的內容（例如：對照投影片〈信心等級〉與實作二的說明）：